In [ ]:
import sys
!conda install -n edumindai xgboost -y

In [2]:
# Import all libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
processed_path = r'C:\Users\Amsha\Desktop\EduMindAI\data\processed'
models_path = r'C:\Users\Amsha\Desktop\EduMindAI\models'

print(" All libraries imported!")

 All libraries imported!


In [3]:
#load processed datasets
df_perf = pd.read_csv(os.path.join(processed_path, 'performance_processed.csv'))
df_ment = pd.read_csv(os.path.join(processed_path, 'mental_processed.csv'))
df_drop = pd.read_csv(os.path.join(processed_path, 'dropout_processed.csv'))
df_beh = pd.read_csv(os.path.join(processed_path, 'behavior_processed.csv'))
print("✅ All processed datasets loaded!")
print(f"📊 Performance: {df_perf.shape}")
print(f"😰 Mental Health: {df_ment.shape}")
print(f"🚨 Dropout: {df_drop.shape}")
print(f"📱 Behavior: {df_beh.shape}")

✅ All processed datasets loaded!
📊 Performance: (649, 34)
😰 Mental Health: (101, 11)
🚨 Dropout: (4424, 36)
📱 Behavior: (480, 17)


In [4]:
#Model-1 Academic Risk Predictor...
print(" Training Academic Risk Model...")
#step-1 Select features & target
features = ['studytime', 'failures', 'absences', 
            'G1', 'G2', 'internet', 'higher',
            'Medu', 'Fedu', 'freetime', 'goout',
            'health', 'romantic', 'famsup']

X = df_perf[features]
y = df_perf['at_risk']

#step-2 Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

#step-3 Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

#step-4 Evaluate
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n Model trained!")
print(f" Accuracy: {accuracy*100:.2f}%")
print(f"\n Detailed Report:")
print(classification_report(y_test, y_pred, 
      target_names=['Safe', 'At Risk']))

 Training Academic Risk Model...
Training samples: 519
Testing samples:  130

 Model trained!
 Accuracy: 93.85%

 Detailed Report:
              precision    recall  f1-score   support

        Safe       0.97      0.97      0.97       115
     At Risk       0.73      0.73      0.73        15

    accuracy                           0.94       130
   macro avg       0.85      0.85      0.85       130
weighted avg       0.94      0.94      0.94       130



In [6]:
# Save Academic Risk Model
model1_path = os.path.join(models_path, 'academic_risk')
os.makedirs(model1_path, exist_ok=True)

# Save model
with open(os.path.join(model1_path, 'rf_academic_risk.pkl'), 'wb') as f:
    pickle.dump(rf_model, f)

# Save feature list
with open(os.path.join(model1_path, 'features.pkl'), 'wb') as f:
    pickle.dump(features, f)

print(" Academic Risk Model saved!")
print(f" Location: {model1_path}")
print(f" Accuracy: {accuracy*100:.2f}%")

 Academic Risk Model saved!
 Location: C:\Users\Amsha\Desktop\EduMindAI\models\academic_risk
 Accuracy: 93.85%


In [10]:
# Model-2  Mental Health Detector...
print(" Training Mental Health Model...")
# Simplify risk to binary
df_ment['mental_risk_binary'] = (df_ment['mental_risk'] >= 2).astype(int)

#step-1 Select features & target
features_ment = ['gender', 'age', 'course', 
                 'year', 'cgpa', 'marital']

X_ment = df_ment[features_ment]
y_ment = df_ment['mental_risk_binary']

#step-2 split-data
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_ment, y_ment, 
    test_size=0.2, 
    random_state=42,
    stratify=y_ment)  

print(f"Training samples: {X_train_m.shape[0]}")
print(f"Testing samples:  {X_test_m.shape[0]}")
print(f"Test class distribution:\n{y_test_m.value_counts()}")

#step-3 Train XGBoost
rf_ment = RandomForestClassifier(
    n_estimators=100, random_state=42)
rf_ment.fit(X_train_m, y_train_m)
#step-4 Evaluate
y_pred_m = rf_ment.predict(X_test_m)
accuracy_m = accuracy_score(y_test_m, y_pred_m)

print(f"\n Model trained!")
print(f" Accuracy: {accuracy_m*100:.2f}%")
print(f"\n Detailed Report:")
print(classification_report(y_test_m, y_pred_m,
      target_names=['Low Risk', 'High Risk'],
      zero_division=0))

 Training Mental Health Model...
Training samples: 80
Testing samples:  21
Test class distribution:
mental_risk_binary
0    15
1     6
Name: count, dtype: int64

 Model trained!
 Accuracy: 61.90%

 Detailed Report:
              precision    recall  f1-score   support

    Low Risk       0.73      0.73      0.73        15
   High Risk       0.33      0.33      0.33         6

    accuracy                           0.62        21
   macro avg       0.53      0.53      0.53        21
weighted avg       0.62      0.62      0.62        21



In [11]:
# Save Mental Health Model
model2_path = os.path.join(models_path, 'mental_health')
os.makedirs(model2_path, exist_ok=True)

with open(os.path.join(model2_path, 'rf_mental_health.pkl'), 'wb') as f:
    pickle.dump(rf_ment, f)

with open(os.path.join(model2_path, 'features.pkl'), 'wb') as f:
    pickle.dump(features_ment, f)

print(" Mental Health Model saved!")
print(f" Location: {model2_path}")
print(f" Accuracy: {accuracy_m*100:.2f}%")

 Mental Health Model saved!
 Location: C:\Users\Amsha\Desktop\EduMindAI\models\mental_health
 Accuracy: 61.90%


In [12]:
# Model-3 Dropout Predictor...
print(" Training Dropout Predictor...")
#step-1 Select features & target
features_drop = [
    'Age at enrollment',
    'Debtor',
    'Tuition fees up to date',
    'Gender',
    'Scholarship holder',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Unemployment rate',
    'Inflation rate',
    'GDP'
]

X_drop = df_drop[features_drop]
y_drop = df_drop['is_dropout']
#step-2 Split data with stratify
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_drop, y_drop,
    test_size=0.2,
    random_state=42,
    stratify=y_drop)

print(f"Training samples: {X_train_d.shape[0]}")
print(f"Testing samples:  {X_test_d.shape[0]}")
#step-3 Train XGBoost
xgb_drop = xgb.XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='logloss')
xgb_drop.fit(X_train_d, y_train_d)
#step-4 Evaluate
y_pred_d = xgb_drop.predict(X_test_d)
accuracy_d = accuracy_score(y_test_d, y_pred_d)

print(f"\n Model trained!")
print(f" Accuracy: {accuracy_d*100:.2f}%")
print(f"\n Detailed Report:")
print(classification_report(y_test_d, y_pred_d,
      target_names=['Not Dropout', 'Dropout'],
      zero_division=0))

 Training Dropout Predictor...
Training samples: 3539
Testing samples:  885

 Model trained!
 Accuracy: 85.31%

 Detailed Report:
              precision    recall  f1-score   support

 Not Dropout       0.88      0.90      0.89       601
     Dropout       0.78      0.75      0.77       284

    accuracy                           0.85       885
   macro avg       0.83      0.83      0.83       885
weighted avg       0.85      0.85      0.85       885

